# Paso a Paso creacion estructuras Catalogo, Schema, Volumenes, Tablas

In [0]:
-- PASO 1
-- creacion de catalogo
create catalog if not exists  bootcamp_de_valentin

In [0]:
show catalogs

In [0]:
-- PASO 2
-- catalgo a usar
USE CATALOG bootcamp_de_valentin;

-- creacion de schemas
create schema if not exists bootcamp_de_valentin.landing
comment 'schema landing para almacenar archivos. Bootcamp Data Engineering - Luciano Argolo'
;
create schema if not exists bootcamp_de_valentin.bronze
comment 'schema bronze para datos crudos en formato string listos para ser procesados y limpiados para Silver. Bootcamp Data Engineering - Luciano Argolo'
;

-- verificacion
show schemas



In [0]:
-- PASO 3
-- creacion de volumenes dentro de Landing
CREATE VOLUME IF NOT EXISTS bootcamp_de_valentin.landing.archivos
COMMENT 'volumen para almacenamiento de archivos crudos en Landing. Bootcamp Data Engineering - Luciano Argolo';

-- verificacion
SHOW VOLUMES IN bootcamp_de_valentin.landing

In [0]:
-- PASO 4
-- verificacion de archivos cargados en el volumen
LIST '/Volumes/bootcamp_de_valentin/landing/archivos/';

In [0]:
-- PASO 5
-- leer archivo de landing
SELECT * FROM read_files(
    '/Volumes/bootcamp_de_valentin/landing/archivos/properties_raw.csv',
    format => 'csv',
    header => true
  )

In [0]:
-- PASO 6
-- creacion de tabla en bronze a partir del archivo de landing, unicamente con los URL valids
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.bronze.propiedades_bronze
  SELECT * FROM read_files(
    '/Volumes/bootcamp_de_valentin/landing/archivos/properties_raw.csv',
    format => 'csv',
    header => true
  )
  WHERE url LIKE 'https%'
  ;

-- verificacion
SHOW TABLES IN bootcamp_de_valentin.bronze

# **EDA propiedades_bronze**

## 1-3 Exploracion Inicial y Analisis de Calidad

### 1 Exploracion inicial

In [0]:
-- ejercicio 1.1
-- Conteo de filas disponibles
SELECT
  format_number(count(*), 2) total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
-- ejercicio 1.2
-- estructura de la tabla
DESCRIBE bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
-- ejercicio 1.3
-- ver muestra de datos
SELECT
  *
FROM bootcamp_de_valentin.bronze.propiedades_bronze
LIMIT 10;

-- Campos de interes
SELECT
  id,
  zona,
  ubicacion,
  precio,
  expensas,
  moneda,
  metros_cuadrados_totales,
  cochera,
  antiguedad,
  estado,
  url
FROM bootcamp_de_valentin.bronze.propiedades_bronze
LIMIT 10;


### 2 Analisis valores nulos

In [0]:
-- ejercicio 2.1
-- conteo de nulos por columna
SELECT
  COUNT(*) total_registros,
  COUNT(*) - COUNT(precio) nulos_precio,
  COUNT(*) - COUNT(expensas) nulos_expensas,
  COUNT(*) - COUNT(tipo_de_operacion) nulos_tipo,
  COUNT(*) - COUNT(moneda) nulos_moneda,
  COUNT(*) - COUNT(ambientes) nulos_ambientes,
  COUNT(*) - COUNT(metros_cuadrados_totales) nulos_m2_totales,
  COUNT(*) - COUNT(metros_cuadrados_cubiertos) nulos_m2_cubiertos,
  COUNT(*) - COUNT(orientacion_cardinal) nulos_orientacion_cardinal,
  COUNT(*) - COUNT(piso) nulos_piso,
  COUNT(*) - COUNT(cochera) nulos_cochera,
  COUNT(*) - COUNT(estado) nulos_estado,
  COUNT(*) - COUNT(antiguedad) nulos_antiguedad,
  COUNT(*) - COUNT(zona) nulos_zona
FROM bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
-- conteo nulos % por columna
with nulos as (
  SELECT
  COUNT(*) total_registros,
  COUNT(*) - COUNT(precio) nulos_precio,
  COUNT(*) - COUNT(expensas) nulos_expensas,
  COUNT(*) - COUNT(tipo_de_operacion) nulos_tipo,
  COUNT(*) - COUNT(moneda) nulos_moneda,
  COUNT(*) - COUNT(ambientes) nulos_ambientes,
  COUNT(*) - COUNT(metros_cuadrados_totales) nulos_m2_totales,
  COUNT(*) - COUNT(metros_cuadrados_cubiertos) nulos_m2_cubiertos,
  COUNT(*) - COUNT(orientacion_cardinal) nulos_orientacion_cardinal,
  COUNT(*) - COUNT(piso) nulos_piso,
  COUNT(*) - COUNT(cochera) nulos_cochera,
  COUNT(*) - COUNT(estado) nulos_estado,
  COUNT(*) - COUNT(antiguedad) nulos_antiguedad,
  COUNT(*) - COUNT(zona) nulos_zona
FROM bootcamp_de_valentin.bronze.propiedades_bronze
)
select
  ROUND( nulos_precio / total_registros * 100, 2 ) porcentaje_nulos_precio,
  ROUND( nulos_expensas / total_registros * 100, 2 ) porcentaje_nulos_expensas,
  ROUND( nulos_tipo / total_registros * 100, 2 ) porcentaje_nulos_tipo,
  ROUND( nulos_moneda / total_registros * 100, 2 ) porcentaje_nulos_moneda,
  ROUND( nulos_ambientes / total_registros * 100, 2 ) porcentaje_nulos_ambientes,
  ROUND( nulos_m2_totales / total_registros * 100, 2 ) porcentaje_nulos_m2_totales,
  ROUND( nulos_m2_cubiertos / total_registros * 100, 2 ) porcentaje_nulos_m2_cubiertos,
  ROUND( nulos_orientacion_cardinal / total_registros * 100, 2 ) porcentaje_nulos_orientacion_cardinal,
  ROUND( nulos_piso / total_registros * 100, 2 ) porcentaje_nulos_piso,
  ROUND( nulos_cochera / total_registros * 100, 2 ) porcentaje_nulos_cochera,
  ROUND( nulos_estado / total_registros * 100, 2 ) porcentaje_nulos_estado,
  ROUND( nulos_antiguedad / total_registros * 100, 2 ) porcentaje_nulos_antiguedad,
  ROUND( nulos_zona / total_registros * 100, 2 ) porcentaje_nulos_zona
from nulos

In [0]:
-- ejercicio 2.3
-- columnas con > 50% nulos
with nulos as (
  SELECT
  COUNT(*) total_registros,
  COUNT(*) - COUNT(precio) nulos_precio,
  COUNT(*) - COUNT(expensas) nulos_expensas,
  COUNT(*) - COUNT(tipo_de_operacion) nulos_tipo,
  COUNT(*) - COUNT(moneda) nulos_moneda,
  COUNT(*) - COUNT(ambientes) nulos_ambientes,
  COUNT(*) - COUNT(metros_cuadrados_totales) nulos_m2_totales,
  COUNT(*) - COUNT(metros_cuadrados_cubiertos) nulos_m2_cubiertos,
  COUNT(*) - COUNT(orientacion_cardinal) nulos_orientacion_cardinal,
  COUNT(*) - COUNT(piso) nulos_piso,
  COUNT(*) - COUNT(cochera) nulos_cochera,
  COUNT(*) - COUNT(estado) nulos_estado,
  COUNT(*) - COUNT(antiguedad) nulos_antiguedad,
  COUNT(*) - COUNT(zona) nulos_zona
FROM bootcamp_de_valentin.bronze.propiedades_bronze
),
pct_nulos AS(
  select
    ROUND( nulos_precio / total_registros * 100, 2 ) porcentaje_nulos_precio,
    ROUND( nulos_expensas / total_registros * 100, 2 ) porcentaje_nulos_expensas,
    ROUND( nulos_tipo / total_registros * 100, 2 ) porcentaje_nulos_tipo,
    ROUND( nulos_moneda / total_registros * 100, 2 ) porcentaje_nulos_moneda,
    ROUND( nulos_ambientes / total_registros * 100, 2 ) porcentaje_nulos_ambientes,
    ROUND( nulos_m2_totales / total_registros * 100, 2 ) porcentaje_nulos_m2_totales,
    ROUND( nulos_m2_cubiertos / total_registros * 100, 2 ) porcentaje_nulos_m2_cubiertos,
    ROUND( nulos_orientacion_cardinal / total_registros * 100, 2 ) porcentaje_nulos_orientacion_cardinal,
    ROUND( nulos_piso / total_registros * 100, 2 ) porcentaje_nulos_piso,
    ROUND( nulos_cochera / total_registros * 100, 2 ) porcentaje_nulos_cochera,
    ROUND( nulos_estado / total_registros * 100, 2 ) porcentaje_nulos_estado,
    ROUND( nulos_antiguedad / total_registros * 100, 2 ) porcentaje_nulos_antiguedad,
    ROUND( nulos_zona / total_registros * 100, 2 ) porcentaje_nulos_zona
  from nulos
)
select
  columna,
  pct_nulos
from pct_nulos
UNPIVOT (
  pct_nulos FOR columna IN (
    porcentaje_nulos_precio,
    porcentaje_nulos_expensas,
    porcentaje_nulos_ambientes,
    porcentaje_nulos_moneda,
    porcentaje_nulos_tipo,
    porcentaje_nulos_m2_totales,
    porcentaje_nulos_m2_cubiertos,
    porcentaje_nulos_orientacion_cardinal,
    porcentaje_nulos_piso,
    porcentaje_nulos_cochera,
    porcentaje_nulos_estado,
    porcentaje_nulos_antiguedad,
    porcentaje_nulos_zona
  )
  -- precio, expensas, ambientes, 
  --       metros_cuadrados_totales, metros_cuadrados_cubiertos,
  --       orientacion_cardinal, antiguedad, piso, cochera
)
WHERE pct_nulos >= 50
ORDER BY pct_nulos DESC


### 3 Cardinalidad y Distribucion

In [0]:
-- ejercicio 3.1
-- distribucion por tipo de operacion
with conteos(
SELECT
  tipo_de_operacion,
  count(*) conteo,
  sum(count(*)) OVER () total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze
GROUP BY tipo_de_operacion
)
select
  tipo_de_operacion,
  conteo,
  ROUND( conteo / total_registros * 100,5) pct
from conteos
order by pct desc

In [0]:
-- ejercicio 3.2
-- distribucion por moneda
with conteos(
SELECT
  moneda,
  count(*) conteo,
  sum(count(*)) OVER () total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze
GROUP BY moneda
)
select
  moneda,
  conteo,
  ROUND( conteo / total_registros * 100,5) pct
from conteos
order by pct desc

In [0]:
-- ejercicio 3.3
-- distribucion por ambientes
with conteos(
SELECT
  ambientes,
  count(*) conteo,
  sum(count(*)) OVER () total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze
GROUP BY ambientes
)
select
  CAST( ambientes as DOUBLE) ambientes,
  conteo,
  ROUND( conteo / total_registros * 100,5) pct
from conteos
order by ambientes desc

In [0]:
-- ejercicio 3.4
-- distribucion por zona
with conteos(
SELECT
  zona,
  count(*) conteo,
  sum(count(*)) OVER () total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze
GROUP BY zona
)
select
  zona,
  conteo,
  ROUND( conteo / total_registros * 100,5) pct
from conteos
order by pct desc
LIMIT 15

In [0]:
-- ejercicio 3.5
-- distribucion por estado
with conteos(
SELECT
  estado,
  count(*) conteo,
  sum(count(*)) OVER () total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze
GROUP BY estado
)
select
  estado,
  conteo,
  ROUND( conteo / total_registros * 100,5) pct
from conteos
order by pct desc

## 4 Estadisticas Descriptivas de Variables Numericas


In [0]:
/*
  Antes de pasar al analisis descriptivo, recordar que los datos son del tipo String, por ende no se pueden aplicar funciones matematicas sobre ellos, pues fallaran.
*/
describe bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
/*
  Ante esto se propone crear una vista temporal (solo vive mientras el cluster este encendido, luego se elimina)
  con los campos numericos formateados correctamente para hacer el analisis
*/

In [0]:
create or replace temporary view propiedades_clean as
select 
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
-- ejercicio 4.1
-- estadisticas de precios por tipo de moneda
SELECT
  moneda,
  count(*) conteo,
  round(avg(precio),2) precio,
  max(precio) max,
  min(precio) min,
  percentile(precio, array(0.25, 0.5, 0.75)) percentiles
FROM propiedades_clean
GROUP BY moneda


Del análisis anterior detectamos:

Monedas basura: MXN, guaranies, uyu, consultar, ar — son errores del scraping (< 15 registros en total)

Tipos de operación sucios: variantes como "alquieler", "alquier", "alguilar", "venta/alquiler", etc.

Nulls: registros sin moneda y sin tipo de operación
Outliers extremos: precios máximos de 1,111,111,111 USD y 1,434,768,228 ARS son claramente errores

Acción: Filtramos solo USD y ARS, solo los 3 tipos de operación principales (venta, alquiler, alquiler_temporal), y usamos percentiles P1/P99 por moneda+operación para cortar outliers extremos.

In [0]:
-- ============================================================
-- Creamos la nueva vista temporal mas limpia
-- ============================================================
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
-- ejercicio 4.2
-- analisis de m2
WITH m2_totales AS(
  SELECT
    'm2_totales' AS tipo,
    avg(metros_cuadrados_totales) promedio,
    percentile(metros_cuadrados_totales,0.5) mediana,
    min(metros_cuadrados_totales) min,
    max(metros_cuadrados_totales) max
  FROM propiedades_clean_2
  WHERE
    metros_cuadrados_totales > 0
),
m2_cubiertos AS(
  SELECT
    'm2_cubiertos' AS tipo,
    avg(metros_cuadrados_cubiertos) promedio,
    percentile(metros_cuadrados_cubiertos,0.5) mediana,
    min(metros_cuadrados_cubiertos) min,
    max(metros_cuadrados_cubiertos) max
  FROM propiedades_clean_2
  WHERE
    metros_cuadrados_cubiertos > 0
)
SELECT * FROM m2_totales
UNION ALL
SELECT * FROM m2_cubiertos

In [0]:
-- ejercicio 4.3
-- analisis de antiguedad
SELECT
  antiguedad,
  count(*) cantidad
FROM propiedades_clean_2
GROUP BY antiguedad
ORDER BY count(*) DESC
LIMIT 20


In [0]:
-- ejercicio 4.4
-- analisis de antiguedad sin placeholder
SELECT
  count(*) cantidad,
  min(antiguedad) min,
  max(antiguedad) max,
  avg(antiguedad) promedio,
  percentile(antiguedad,0.5) mediana
FROM propiedades_clean_2
WHERE
  antiguedad <> 999
  and antiguedad is not NULL
  and antiguedad >= 0


## 5 Deteccion de Problemas de Calidad

In [0]:
WITH
cantidad AS(
  SELECT
    'Cantidad Registros' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
),
precios AS(
  SELECT
    'Precios' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    precio is NULL
    or precio <= 0
),
m2_totales AS(
  SELECT
    'm2_totales' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    metros_cuadrados_totales is NULL
    or metros_cuadrados_totales <= 0
),
m2_cubiertos AS(
  SELECT
    'm2_cubiertos' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    metros_cuadrados_cubiertos is NULL
    or metros_cuadrados_cubiertos <= 0
),
antiguedad AS(
  SELECT
    'antiguedad' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    antiguedad = 999
),
ambientes AS(
  SELECT
    'ambientes' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    ambientes is NULL
    or ambientes <= 0
),
monedas AS(
  SELECT
    'monedas' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    moneda is NULL
),
tipo_operacion AS(
  SELECT
    'tipo_operacion' Tipo_Analisis,
    COUNT(*) Conteo
  FROM propiedades_clean_2
  WHERE
    tipo_de_operacion is NULL
),
uniones as (
-- SELECT * FROM cantidad
-- UNION ALL
SELECT * FROM precios
UNION ALL
SELECT * FROM m2_totales
UNION ALL
SELECT * FROM m2_cubiertos
UNION ALL
SELECT * FROM antiguedad
UNION ALL
SELECT * FROM ambientes
UNION ALL
SELECT * FROM monedas
UNION ALL
SELECT * FROM tipo_operacion
)
SELECT
 u.*,
 round( u.conteo /  c.conteo * 100,2) as pct
FRoM uniones u
cross join cantidad c

In [0]:
-- ejercicio 5.2
-- deteccion de duplicados
with duplicados as(
  SELECT
    precio,
    url,
    count(*) conteo
  FROM bootcamp_de_valentin.bronze.propiedades_bronze
  group by precio, url
  having count(*) > 1
)
SELECT
  count(*) grupos_duplicados,
  sum(conteo) registros_duplicados,
  sum(conteo - 1) registros_extras_por_dupliacion
FROM duplicados

In [0]:
-- ejercicio 5.3
-- ejemplos de duplicados
SELECT
    precio,
    url,
    count(*) conteo
  FROM bootcamp_de_valentin.bronze.propiedades_bronze
  group by precio, url
  having count(*) > 1
  order by conteo desc
  limit 10

In [0]:
-- ejercicio 5.4
-- outliers en precio
WITH percentiles as (
  SELECT
    moneda,
    percentile(precio, 0.01) p01,
    percentile(precio, 0.99) p99
  FROM propiedades_clean_2
  WHERE precio > 0
  group by moneda

)
-- SELECT * from percentiles 500 - 2.200.000
SELECT
  p.moneda,
  'muy alto > p.99' tipo_outlier,
  count(*) conteo
FROM propiedades_clean_2 p
JOIn percentiles pct ON p.moneda = pct.moneda
WHERE p.precio > pct.p99
GROUP BY p.moneda
UNION ALL
SELECT
  p.moneda,
  'muy bajo < p.01' tipo_outlier,
  count(*) conteo
FROM propiedades_clean_2 p
JOIn percentiles pct ON p.moneda = pct.moneda
WHERE p.precio < pct.p01
GROUP BY p.moneda

## 6-7 Analisis Avanzado y Documentacion

In [0]:
create or replace temp view bronze_EDA as (
SELECT 
    edl.id,
    edl.ubicacion,
    CASE 
        WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.precio::float BETWEEN -2147483648 AND 2147483648 THEN edl.precio::float
        ELSE NULL
    END AS precio,
    CASE 
        WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
        ELSE NULL
    END AS numero,
    edl.calle,
    CASE
        WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.expensas::float BETWEEN -20000000 AND 20000000 THEN edl.expensas::float
        ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
        WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%mxn%' THEN 'MXN'
        WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
        WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
        ELSE edl.moneda
    END AS moneda,
    CASE
        WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.ambientes::float
    END AS ambientes,
    CASE 
        WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE 
        WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
        WHEN edl.piso IS NULL THEN NULL
        WHEN edl.piso = 'NaN' THEN NULL
        ELSE edl.piso::float
    END AS piso,
    CASE 
        WHEN edl.cochera = 'tiene' THEN 1
        ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    edl.fecha,
    edl.hora
FROM propiedades_clean_2 edl
);

In [0]:
-- ejercicio 6.1
-- ranking de zonas por precio
SELECT
  zona,
  round( avg(precio),2) precio_promedio_zona,
  count(*) cantidad_propiedades,
  row_number() over ( order by round( avg(precio),2) desc) ranking
FROM bronze_eda
WHERE
  moneda = 'ARS'
  and tipo_de_operacion = 'alquiler'
GROUP by zona
QUALIFY ranking <= 10

In [0]:
-- ejercicio 6.2
-- Comparacion con promedio general
WITH por_zona(
SELECT
  zona,
  round( avg(precio),2) precio_promedio_zona
FROM bronze_eda
WHERE
  moneda = 'ARS'
  and tipo_de_operacion = 'alquiler'
GROUP by zona
)
SELECT
  zona,
  precio_promedio_zona,
  ROUND( AVG(precio_promedio_zona) OVER (),2) promedio_general,
  ROUND( precio_promedio_zona - AVG(precio_promedio_zona) OVER (),2) diferencia,
  ROUND( (precio_promedio_zona - AVG(precio_promedio_zona) OVER ()) / AVG(precio_promedio_zona) OVER () * 100 ,2) diferencia_pct
FROM por_zona

In [0]:
-- ejercicio 6.3
-- analisis temporal
WITH promedios as (
SELECT
  Cast( date_trunc('month', fecha) as date) fecha_mes,
  round( avg(precio),2) precio_promedio
FROM bronze_eda
WHERE 
  moneda = 'ARS'
  and tipo_de_operacion = 'alquiler'
GROUP BY date_trunc('month', fecha)
ORDER BY date_trunc('month',fecha) 
),
anterior as (
SELECT
  *,
  lag(precio_promedio) over (order by fecha_mes) precio_promedio_anterior
from promedios
)
SELECT
  *,
  precio_promedio - precio_promedio_anterior diferencia,
  (precio_promedio - precio_promedio_anterior) / precio_promedio_anterior * 100 diferencia_pct
from anterior

In [0]:
-- ejercicio 7.1
-- Resumen Ejecutivo

**Resumen ejecutivo:**

- Total de registros analizados: 509,395 propiedades
- Período: julio 2025 - enero 2026
- Zonas únicas: 98
- Datos válidos (precio > 0): ~99.6% — la mayoría de registros tienen precio
- Datos con problemas: expensas (77.8% nulos), orientación (96.1% nulos), antigüedad (98.9% con valor 999)

Problemas de calidad identificados:
- Valores placeholder: El valor 999 en antigüedad representa datos faltantes
- Campos nulos: Varios campos como expensas, orientacion, cochera tienen alto % de nulos
- Datos mixtos: Precios en USD y ARS mezclados (requiere normalización)
- Posibles duplicados: Propiedades repetidas con misma ubicación y precio
- Outliers: Precios extremadamente altos o bajos que podrían ser errores

Transformaciones necesarias para capa Silver:
- Convertir 999 en antigüedad a NULL
- Normalizar precios a una sola moneda (o crear columnas separadas)
- Eliminar duplicados
- Filtrar outliers extremos
- Parsear el campo ubicación para extraer barrio/ciudad
- Calcular métricas derivadas (precio por m2)
